# [Numba](https://numba.pydata.org/numba-doc/latest/user/5minguide.html)

Numba is a just-in-time compiler for Python that works best on code that uses NumPy arrays and functions, and loops.

This jupyter notebook gives a short demo on how to use Numba to speed up a function. For instance, cosine similarity is a function that takes two vectors and returns a scalar. Lets try to speed up this function.

In [1]:
from numba import njit
import numpy as np
import time
from tqdm import tqdm

### Simple cosine similairty function

THe formula for cosine similarity is:

$Cos(x, y) = \frac{x . y} {||x|| * ||y||}$

where $x$ and $y$ are vectors, and $||x||$ is the norm of $x$.

Lets define a function that computes the cosine similarity between two vectors in python.

In [2]:

def cosine_similarity(u:np.ndarray, v:np.ndarray):
    """ Computes the cosine similarity between two vectors

    Args:
        u (np.ndarray): vector 1
        v (np.ndarray): vector 2

    Returns:
        _type_: float - cosine similarity
    """
    assert(u.shape[0] == v.shape[0])

    uv = 0
    uu = 0
    vv = 0
    for i in range(u.shape[0]):
        uv += u[i]*v[i]
        uu += u[i]*u[i]
        vv += v[i]*v[i]
    cos_theta = 1
    if uu!=0 and vv!=0:
        cos_theta = uv/np.sqrt(uu*vv)
    return cos_theta

### Sample embedings

Lets generate some sample embeddings using numpy pkg. We will use the `np.random.rand` function to generate random embeddings. 

Once we have the embeddings we will find the cosine similarity between two embeddings in a loop, and create a matrix of cosine similarities.

In [3]:
# generate two random embeddings lists
embed_size = 1024
n_embeds = 500

embeddings_a = np.random.rand(n_embeds, embed_size)
embeddings_b = np.random.rand(n_embeds, embed_size)

# create a empty matrix to store the cosine similarities

similarity_matrix = np.zeros((n_embeds, n_embeds))

 Lets find the time it takes to compute the cosine similarity matrix

In [4]:
start = time.time()

for i in tqdm(range(n_embeds)):
    for j in range(n_embeds):
        similarity_matrix[i, j] = cosine_similarity(embeddings_a[i], embeddings_b[j])
        
end = time.time()
print("Time taken without numba:", end-start, "seconds")

100%|██████████| 500/500 [03:54<00:00,  2.13it/s]

Time taken without numba: 234.5066909790039 seconds


### Cosine similarity matrix generation using Numba

It takes a while to compute the cosine similarity matrix. Lets use Numba to speed up the computation.

Numba package provides decorators that allow us to speed up the computation. 

In [5]:
@njit(fastmath=True)
def cosine_similarity_numba(u:np.ndarray, v:np.ndarray):
    """ Computes the cosine similarity between two
        vectors using numba.

    Args:
        u (np.ndarray): vector 1
        v (np.ndarray): vector 2

    Returns:
        _type_: float - cosine similarity
    """
    assert(u.shape[0] == v.shape[0])

    uv = 0
    uu = 0
    vv = 0
    for i in range(u.shape[0]):
        uv += u[i]*v[i]
        uu += u[i]*u[i]
        vv += v[i]*v[i]
    cos_theta = 1
    if uu!=0 and vv!=0:
        cos_theta = uv/np.sqrt(uu*vv)
    return cos_theta

In [6]:
start = time.time()

for i in tqdm(range(n_embeds)):
    for j in range(n_embeds):
        similarity_matrix[i, j] = cosine_similarity_numba(embeddings_a[i], embeddings_b[j])
        
end = time.time()
print("Time taken with numba:", end-start, "seconds")

100%|██████████| 500/500 [00:01<00:00, 260.95it/s]

Time taken with numba: 1.9230973720550537 seconds


From the above example, we can see that numba computed the cosine similarity matrix way faster than the python version. This is because numba compiles the code to machine code, and then executes the machine code. 

It takes more time for the first run using numba as during the first run the code is compiled to machine code. (Still way faster than the python version)

Once the code is compiled, it is faster to execute the machine code than the python code. You can see the speedup in below run. 

In [7]:
start = time.time()

for i in tqdm(range(n_embeds)):
    for j in range(n_embeds):
        similarity_matrix[i, j] = cosine_similarity_numba(embeddings_a[i], embeddings_b[j])
        
end = time.time()
print("Time taken with numba after compilation:", end-start, "seconds")

100%|██████████| 500/500 [00:00<00:00, 1263.40it/s]

Time taken with numba after compilation: 0.3995654582977295 seconds
